# Sea Ice Albedo — BioSNICAR MVP Demo

This notebook demonstrates the sea-ice extension to BioSNICAR (v0.1).  
We show:
1. Running a preset and plotting the spectral albedo
2. Comparing FYI, MYI, and snow-covered conditions
3. Sensitivity to salinity, temperature, and solar zenith angle
4. Building a custom column from scratch
5. Known limitations

**Reference for sea ice physics**: Light et al. (2004), Perovich et al. (2002), Cox & Weeks (1983)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from biosnicar.sea_ice.api import SeaIceColumn, SeaIceLayer, SnowLayer
from biosnicar.sea_ice.presets import FYI_WINTER_BARE, FYI_WINTER_SNOW, MYI_WINTER_BARE

plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})

## 1. Preset: First-Year Ice (bare, winter)

The `FYI_WINTER_BARE` preset represents typical Arctic first-year ice in winter:  
- Surface layer: T=−25°C, S=12 psu, ρ=920 kg/m³  
- Bulk layer: T=−10°C, S=8 psu, ρ=915 kg/m³  

In [ ]:
col_fyi = SeaIceColumn.from_preset(FYI_WINTER_BARE)
result_fyi = col_fyi.compute_albedo(sza_deg=60, atmosphere='sub_arctic_winter')

print(f"FYI bare  — BBA={result_fyi.broadband:.3f}, VIS={result_fyi.visible:.3f}, NIR={result_fyi.nir:.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(result_fyi.wavelengths, result_fyi.spectrum, lw=1.5, label='FYI bare (SZA=60°)')
ax.set_xlabel('Wavelength (µm)')
ax.set_ylabel('Spectral albedo')
ax.set_xlim(0.3, 2.5)
ax.set_ylim(0, 1.05)
ax.axvline(0.75, color='k', ls='--', lw=0.7, alpha=0.4, label='VIS/NIR boundary')
ax.legend()
ax.set_title('Sea ice spectral albedo — BioSNICAR v0.1')
plt.tight_layout()
plt.show()

## 2. Comparing FYI, MYI, and snow-covered conditions

In [ ]:
result_snow = SeaIceColumn.from_preset(FYI_WINTER_SNOW).compute_albedo(sza_deg=60)
result_myi = SeaIceColumn.from_preset(MYI_WINTER_BARE).compute_albedo(sza_deg=60)

fig, ax = plt.subplots(figsize=(9, 5))
wvl = result_fyi.wavelengths

ax.plot(wvl, result_snow.spectrum,  lw=1.8, label=f'FYI + snow  BBA={result_snow.broadband:.2f}', color='steelblue')
ax.plot(wvl, result_myi.spectrum,   lw=1.8, label=f'MYI bare    BBA={result_myi.broadband:.2f}', color='darkorange')
ax.plot(wvl, result_fyi.spectrum,   lw=1.8, label=f'FYI bare    BBA={result_fyi.broadband:.2f}', color='firebrick')

ax.set_xlabel('Wavelength (µm)')
ax.set_ylabel('Spectral albedo')
ax.set_xlim(0.3, 2.5)
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title('Sea ice types — SZA=60°, sub-Arctic winter')
plt.tight_layout()
plt.show()

print("Broadband albedo summary:")
for name, r in [('FYI bare', result_fyi), ('MYI bare', result_myi), ('FYI+snow', result_snow)]:
    print(f"  {name:12s}  BBA={r.broadband:.3f}  VIS={r.visible:.3f}  NIR={r.nir:.3f}")

## 3. Sensitivity analysis

### 3a. Effect of salinity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Salinity sweep
salinities = [1, 4, 8, 12]
ax = axes[0]
for S in salinities:
    col = SeaIceColumn(layers=[
        SeaIceLayer(thickness_m=1.5, temperature_C=-10, salinity_psu=S,
                    density_kg_m3=910, bubble_radius_um=300)
    ])
    r = col.compute_albedo(sza_deg=60)
    ax.plot(r.wavelengths, r.spectrum, lw=1.5, label=f'S={S} psu  (BBA={r.broadband:.2f})')
ax.set_xlim(0.3, 2.5); ax.set_ylim(0, 1.05)
ax.set_xlabel('Wavelength (µm)'); ax.set_ylabel('Albedo')
ax.set_title('Salinity sensitivity (T=−10°C)')
ax.legend(fontsize=9)

# Temperature sweep
temperatures = [-2, -5, -10, -20]
ax = axes[1]
for T in temperatures:
    col = SeaIceColumn(layers=[
        SeaIceLayer(thickness_m=1.5, temperature_C=T, salinity_psu=8,
                    density_kg_m3=910, bubble_radius_um=300)
    ])
    r = col.compute_albedo(sza_deg=60)
    ax.plot(r.wavelengths, r.spectrum, lw=1.5, label=f'T={T}°C  (BBA={r.broadband:.2f})')
ax.set_xlim(0.3, 2.5); ax.set_ylim(0, 1.05)
ax.set_xlabel('Wavelength (µm)'); ax.set_ylabel('Albedo')
ax.set_title('Temperature sensitivity (S=8 psu)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### 3b. Effect of solar zenith angle

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for sza in [30, 45, 60, 75]:
    col = SeaIceColumn.from_preset(FYI_WINTER_BARE)
    r = col.compute_albedo(sza_deg=sza)
    ax.plot(r.wavelengths, r.spectrum, lw=1.5, label=f'SZA={sza}°  (BBA={r.broadband:.2f})')

ax.set_xlim(0.3, 2.5); ax.set_ylim(0, 1.05)
ax.set_xlabel('Wavelength (µm)'); ax.set_ylabel('Spectral albedo')
ax.set_title('Solar zenith angle sensitivity — FYI bare')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Custom column

Build a 3-layer column with a thin snow cover and two sea-ice layers:

In [ ]:
custom_col = SeaIceColumn(layers=[
    # Fresh snow at top (MVP: salty snow deferred to v0.2)
    SnowLayer(thickness_m=0.08, density_kg_m3=280, grain_radius_um=150),
    # Cold saline surface sea ice
    SeaIceLayer(
        thickness_m=0.10,
        temperature_C=-22,
        salinity_psu=10,
        density_kg_m3=918,
        bubble_radius_um=120,
        layer_class="FYI",
    ),
    # Warmer bulk sea ice
    SeaIceLayer(
        thickness_m=1.80,
        temperature_C=-8,
        salinity_psu=5,
        density_kg_m3=912,
        bubble_radius_um=250,
        layer_class="FYI",
    ),
])

r_custom = custom_col.compute_albedo(sza_deg=65, atmosphere='sub_arctic_winter', sky='clear')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(r_custom.wavelengths, r_custom.spectrum, lw=2, color='teal')
ax.set_xlim(0.3, 2.5); ax.set_ylim(0, 1.05)
ax.set_xlabel('Wavelength (µm)'); ax.set_ylabel('Spectral albedo')
ax.set_title(f'Custom 3-layer column  (BBA={r_custom.broadband:.3f}, SZA=65°)')
plt.tight_layout()
plt.show()

## 5. Comparison with approximate SHEBA observations

Perovich et al. (2002) report typical Arctic sea-ice broadband albedos during winter:  
- Bare FYI: ~0.40–0.55  
- Snow-covered ice: ~0.75–0.90  
- MYI: ~0.50–0.65  

Our model values fall within or near these ranges.

In [ ]:
obs_ranges = {
    'FYI bare':   (0.40, 0.55),
    'FYI + snow': (0.75, 0.90),
    'MYI bare':   (0.50, 0.65),
}
model_bbas = {
    'FYI bare':   result_fyi.broadband,
    'FYI + snow': result_snow.broadband,
    'MYI bare':   result_myi.broadband,
}

print(f"{'Condition':<15}  {'Observed BBA':^18}  {'Model BBA':^10}  {'In range?':^10}")
print('-' * 60)
for k in obs_ranges:
    lo, hi = obs_ranges[k]
    bba = model_bbas[k]
    in_range = lo <= bba <= hi
    print(f"{k:<15}  [{lo:.2f}–{hi:.2f}]            {bba:.3f}       {'✓' if in_range else '~'}")

## 6. Known limitations of v0.1

- **Snow on sea ice is fresh water** — salty snow and brine wicking are deferred to v0.2.
- **Brine RI approximation** — calibrated for seawater at ~35 psu; extrapolated to brine at up to ~200 psu.
- **Maxwell-Garnett sphere assumption** — valid for brine volume fractions < ~0.3.
- **Constant T and S per layer** — no vertical gradients within a layer.
- **No sea-ice algae** — deferred to v0.2.
- **No melt ponds** — deferred to v0.2.
- **Arctic validation only** — Antarctic sea ice has not been tuned.

See `docs/sea_ice.md` for the full documentation.